# Análisis SBOM — Software Bill of Materials

Syft genera un inventario completo de dependencias (SBOM) en formato CycloneDX
para cada repositorio. Este notebook analiza la composición de software:
ecosistemas de paquetes, licencias, componentes compartidos y concentración de dependencias.

**Fuente de datos:** colecciones `sbom_components` y `sbom_scans` en `secpipeline.json`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DB_PATH = os.getenv("DB_PATH", "/data/secpipeline.json")
if not Path(DB_PATH).exists():
    DB_PATH = str(Path("../data/secpipeline.json").resolve())

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

with open(DB_PATH) as f:
    db = json.load(f)

repos  = pd.DataFrame(db.get("repositories", []))
scans  = pd.DataFrame(db.get("sbom_scans", []))
df     = pd.DataFrame(db.get("sbom_components", []))

repo_names = repos.set_index("id")["full_name"].to_dict() if not repos.empty else {}

print(f"Repositorios con SBOM generado : {len(scans)}")
print(f"Total de componentes detectados: {len(df)}")

if df.empty:
    print("\nNo hay componentes SBOM todavía. Ejecutá el miner primero.")
else:
    df["repo_name"] = df["repo_id"].map(repo_names)
    df["ecosystem"] = df["ecosystem"].fillna("desconocido")
    df["license"] = df["license"].fillna("sin licencia")
    print(f"Ecosistemas únicos            : {df['ecosystem'].nunique()}")
    print(f"Componentes únicos (nombre)   : {df['name'].nunique()}")
    print(f"Licencias distintas           : {df['license'].nunique()}")

## 1. Componentes por Ecosistema

El ecosistema identifica el gestor de paquetes: `npm`, `pypi`, `maven`, `gem`, `cargo`, etc.

In [ ]:
if not df.empty:
    eco_counts = df["ecosystem"].value_counts().head(15)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = sns.color_palette("tab10", len(eco_counts))
    axes[0].barh(eco_counts.index[::-1], eco_counts.values[::-1], color=colors[::-1])
    axes[0].set_xlabel("Cantidad de componentes")
    axes[0].set_title("Componentes por ecosistema (top 15)")
    for i, v in enumerate(eco_counts.values[::-1]):
        axes[0].text(v + 0.5, i, str(v), va="center", fontsize=9)

    top_eco = eco_counts.head(8)
    other_count = eco_counts[8:].sum()
    if other_count > 0:
        top_eco = pd.concat([top_eco, pd.Series({"otros": other_count})])
    axes[1].pie(top_eco, labels=top_eco.index, autopct="%1.1f%%",
                startangle=90, colors=sns.color_palette("tab10", len(top_eco)))
    axes[1].set_title("Proporción por ecosistema")

    plt.suptitle("Distribución de ecosistemas de paquetes", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 2. Distribución de Licencias

Las licencias de software determinan los términos de uso. Licencias como GPL pueden
generar obligaciones legales que otras (MIT, Apache) no imponen.

In [ ]:
if not df.empty:
    lic_counts = df["license"].value_counts().head(20)

    # Clasificar licencias por tipo
    def classify_license(name):
        name_l = str(name).lower()
        if any(x in name_l for x in ["mit", "apache", "bsd", "isc", "zlib", "public domain"]):
            return "Permisiva"
        if any(x in name_l for x in ["gpl", "lgpl", "agpl", "copyleft"]):
            return "Copyleft"
        if "commercial" in name_l or "proprietary" in name_l:
            return "Comercial"
        if "sin licencia" in name_l or name_l == "none":
            return "Sin licencia"
        return "Otra"

    df["license_type"] = df["license"].apply(classify_license)
    lic_type_counts = df["license_type"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(lic_counts.index[::-1], lic_counts.values[::-1],
                 color=sns.color_palette("tab20", len(lic_counts))[::-1])
    axes[0].set_xlabel("Componentes")
    axes[0].set_title("Top 20 licencias más frecuentes")

    type_colors = {"Permisiva": "#2ca02c", "Copyleft": "#ff7f0e",
                   "Comercial": "#d62728", "Sin licencia": "#c7c7c7", "Otra": "#9467bd"}
    axes[1].pie(lic_type_counts,
                labels=lic_type_counts.index,
                colors=[type_colors.get(l, "#c7c7c7") for l in lic_type_counts.index],
                autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Clasificación por tipo de licencia")

    plt.suptitle("Análisis de licencias de dependencias", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 3. Componentes por Repositorio

Repositorios con más dependencias tienen mayor superficie de ataque.

In [ ]:
if not df.empty:
    comp_by_repo = df.groupby("repo_name").size().sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(12, max(5, len(comp_by_repo) * 0.4)))
    ax.barh(comp_by_repo.index[::-1], comp_by_repo.values[::-1], color="#9467bd")
    ax.set_xlabel("Cantidad de componentes")
    ax.set_title("Top 20 repositorios por cantidad de dependencias",
                 fontsize=13, fontweight="bold")
    for i, v in enumerate(comp_by_repo.values[::-1]):
        ax.text(v + 0.5, i, str(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nEstadísticas de componentes por repositorio:")
    print(df.groupby("repo_name").size().describe().round(1).to_string())

## 4. Componentes Compartidos Entre Repositorios

Dependencias presentes en múltiples repos representan riesgo sistémico:
una vulnerabilidad en ellas afecta a toda la organización.

In [ ]:
if not df.empty:
    # Componentes presentes en más de un repo
    shared = df.groupby("name")["repo_id"].nunique().sort_values(ascending=False)
    shared_multi = shared[shared > 1]

    print(f"Componentes únicos en total          : {len(shared)}")
    print(f"Componentes compartidos (≥2 repos)   : {len(shared_multi)}")
    print(f"Componentes exclusivos (1 repo)       : {(shared == 1).sum()}")

    if not shared_multi.empty:
        top_shared = shared_multi.head(20)

        fig, ax = plt.subplots(figsize=(11, 7))
        ax.barh(top_shared.index[::-1], top_shared.values[::-1], color="#8c564b")
        ax.set_xlabel("Repositorios que lo utilizan")
        ax.set_title("Top 20 componentes compartidos entre repositorios",
                     fontsize=13, fontweight="bold")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        for i, v in enumerate(top_shared.values[::-1]):
            ax.text(v + 0.1, i, str(v), va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

## 5. Ecosistemas por Repositorio (Heatmap)

Visualiza qué ecosistemas usa cada repositorio, revelando la diversidad tecnológica.

In [ ]:
if not df.empty:
    top_repos = df["repo_name"].value_counts().head(20).index
    top_ecos = df["ecosystem"].value_counts().head(10).index

    heat_data = (
        df[df["repo_name"].isin(top_repos) & df["ecosystem"].isin(top_ecos)]
        .groupby(["repo_name", "ecosystem"])
        .size()
        .unstack(fill_value=0)
    )

    if not heat_data.empty:
        fig, ax = plt.subplots(figsize=(13, max(6, len(heat_data) * 0.4)))
        sns.heatmap(
            heat_data,
            annot=True,
            fmt="d",
            cmap="Blues",
            ax=ax,
            linewidths=0.5,
            cbar_kws={"label": "Componentes"},
        )
        ax.set_title("Componentes por repositorio y ecosistema (top 20 repos)",
                     fontsize=13, fontweight="bold")
        ax.set_ylabel("Repositorio")
        ax.set_xlabel("Ecosistema")
        plt.tight_layout()
        plt.show()